<a id="transfer-function"></a>

$$
\text{传递函数 (Transfer Function): 正弦稳态下, 输出相量与输入相量之比} \\
H(j\omega) = \frac{\mathbf{V}_{out}}{\mathbf{V}_{in}} \text{, } |H| \text{ 是振幅放大倍数 (增益 Gain), } \angle H \text{ 是相位差} \\
\text{输入 } A\cos(\omega t) \implies \text{稳态输出 } A |H(j\omega)| \cos(\omega t + \angle H(j\omega)) \\ \quad \\
\text{RC 低通滤波器 (Low-Pass Filter): R 串联 C, 输出取电容两端} \\
\text{阻抗分压 (和直流分压公式一样, 只是阻抗是复数):} \\
H(j\omega) = \frac{Z_C}{R + Z_C} = \frac{\frac{1}{j\omega C}}{R + \frac{1}{j\omega C}} \xrightarrow{\text{分子分母同乘 } j\omega C} \frac{1}{1 + j\omega RC} = \frac{1}{1 + j\omega\tau} \\
|H| = \frac{1}{|1 + j\omega\tau|} = \frac{1}{\sqrt{1 + (\omega\tau)^2}}, \quad \angle H = -\angle(1 + j\omega\tau) = -\arctan(\omega\tau) \\
\text{和 RC 正弦输入时解 ODE 得到的稳态振幅, 相位完全一样, 但这里只用了复数代数}
$$

数学基础: [欧拉公式](../math_basics/calculus.ipynb#taylor) · 电路: [阻抗](phasor_and_impedance.ipynb#impedance) · [RC 正弦输入的 ODE 解](rc_rl_circuit.ipynb#rc-sine)

<a id="cutoff"></a>

$$
\text{截止频率 (Cutoff Frequency) 与分贝 (Decibel)} \\ \quad \\
\text{分贝: 功率比取 } 10\log_{10} \text{, 功率} \propto \text{电压}^2 \text{, 所以电压比 } |H|_{dB} = 10\log_{10}|H|^2 = 20\log_{10}|H| \\
\text{截止频率定义为输出功率降到一半: } |H|^2 = \frac{1}{2} \implies 1 + (\omega\tau)^2 = 2 \implies \omega_c = \frac{1}{\tau} = \frac{1}{RC}, \ f_c = \frac{1}{2\pi RC} \\
|H(j\omega_c)|_{dB} = 20\log_{10}\frac{1}{\sqrt{2}} = -10\log_{10}2 \approx -3.01 \text{ dB (所以也叫 -3 dB 频率)}, \quad \angle H(j\omega_c) = -\arctan 1 = -45^\circ \\ \quad \\
\text{渐近线 (Asymptotes), 波特图 (Bode Plot) 的横轴是 } \log_{10}\omega \text{:} \\
\omega \ll \omega_c \text{: } |H| \approx 1 \implies 0 \text{ dB (水平线), 相位} \approx 0^\circ \\
\omega \gg \omega_c \text{: } |H| \approx \frac{1}{\omega\tau} \implies |H|_{dB} \approx -20\log_{10}\omega - 20\log_{10}\tau \\
\text{对 } \log_{10}\omega \text{ 是一条斜率为 -20 的直线: 频率每增加 10 倍 (一个十倍频程 decade), 增益下降 20 dB} \\
\text{(频率翻倍 (一个倍频程 octave), 下降 } 20\log_{10}2 \approx 6 \text{ dB), 相位} \to -90^\circ \\
\text{两条渐近线在 } \omega = \omega_c \text{ 相交, 真实曲线在那里比渐近线低 3 dB}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

R, Cap = 1e3, 1e-6
tau = R * Cap
w_c = 1 / tau
print(f'截止角频率 w_c = 1/RC = {w_c:.1f} rad/s, f_c = {w_c / (2 * np.pi):.2f} Hz')

w = np.logspace(1, 6, 2000)
H_lp = 1 / (1 + 1j * w * tau)
mag_db = 20 * np.log10(np.abs(H_lp))
phase = np.rad2deg(np.angle(H_lp))

# scipy: H(s) = 1 / (tau s + 1), freqs 计算 s = jw 处的值
w_s, H_s = signal.freqs([1], [tau, 1], worN=w)
w_b, mag_b, phase_b = signal.bode(signal.lti([1], [tau, 1]), w=w)
print(f'手算 H == scipy.freqs ? {np.allclose(H_lp, H_s)}, 幅度 dB == scipy.bode ? {np.allclose(mag_db, mag_b)}, 相位 == ? {np.allclose(phase, phase_b)}')

H_c = 1 / (1 + 1j * w_c * tau)
print(f'截止频率处: |H| = {abs(H_c):.6f} (1/sqrt2 = {1 / np.sqrt(2):.6f}), {20 * np.log10(abs(H_c)):.4f} dB, 相位 = {np.rad2deg(np.angle(H_c)):.2f} 度')
for k in [10, 100, 1000]:
    print(f'w = {k:>4} w_c: {20 * np.log10(abs(1 / (1 + 1j * k * w_c * tau))):.4f} dB (渐近线 {-20 * np.log10(k):.0f} dB)')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].semilogx(w, mag_db, label='|H| (dB)')
axs[0].semilogx(w, np.where(w < w_c, 0, -20 * np.log10(w / w_c)), 'k--', label='asymptotes')
axs[0].axvline(w_c, color='r', linestyle=':', label='w_c')
axs[0].set_ylim(-60, 5)
axs[0].set_xlabel('omega (rad/s)')
axs[1].semilogx(w, phase, label='phase (deg)')
axs[1].axhline(-45, color='r', linestyle=':', label='-45 deg')
axs[1].set_xlabel('omega (rad/s)')
plot_nicely()
plt.show()

<a id="high-pass"></a>

$$
\text{RC 高通滤波器 (High-Pass Filter): 输出取电阻两端} \\
H(j\omega) = \frac{R}{R + \frac{1}{j\omega C}} = \frac{j\omega RC}{1 + j\omega RC} = \frac{j\omega\tau}{1 + j\omega\tau} \\
|H| = \frac{\omega\tau}{\sqrt{1 + (\omega\tau)^2}}, \quad \angle H = 90^\circ - \arctan(\omega\tau) \\
\text{低频: } |H| \approx \omega\tau \text{, +20 dB/decade (电容隔直流); 高频: } |H| \to 1 \\
\text{截止频率同样是 } \omega_c = \frac{1}{RC} \text{, 并且 } H_{LP} + H_{HP} = \frac{1 + j\omega\tau}{1 + j\omega\tau} = 1 \text{ (KVL: 两个元件电压之和 = 输入)} \\ \quad \\
\text{RLC 带通滤波器 (Band-Pass Filter): 串联 RLC, 输出取电阻两端} \\
H(j\omega) = \frac{R}{R + j\omega L + \frac{1}{j\omega C}} = \frac{R}{R + j\left(\omega L - \frac{1}{\omega C}\right)} \\
\text{在 } \omega_0 = \frac{1}{\sqrt{LC}} \text{ 处虚部为 0, } |H| = 1 \text{ (0 dB)}; \text{ -3 dB 带宽 } \Delta\omega = \frac{R}{L} \text{ (推导见 RLC 谐振)} \\
\text{写成 } s = j\omega \text{ 的有理函数 (分子分母同乘 } \frac{j\omega}{L} \text{): } H(s) = \frac{\frac{R}{L} s}{s^2 + \frac{R}{L} s + \frac{1}{LC}} \\
\text{低频 } \approx \frac{R C s}{1} \text{ (+20 dB/dec), 高频 } \approx \frac{R}{L s} \text{ (-20 dB/dec)}
$$

电路: [RLC 谐振与带宽](rlc_circuit.ipynb#resonance)

In [ ]:
H_hp = 1j * w * tau / (1 + 1j * w * tau)
print(f'高通: 手算 == scipy.freqs ? {np.allclose(H_hp, signal.freqs([tau, 0], [tau, 1], worN=w)[1])}, H_LP + H_HP == 1 ? {np.allclose(H_lp + H_hp, 1)}')

L_bp, C_bp, R_bp = 10e-3, 1e-6, 20.0
w0 = 1 / np.sqrt(L_bp * C_bp)
H_bp = R_bp / (R_bp + 1j * (w * L_bp - 1 / (w * C_bp)))
H_bp_scipy = signal.freqs([R_bp / L_bp, 0], [1, R_bp / L_bp, 1 / (L_bp * C_bp)], worN=w)[1]
mag_bp = np.abs(H_bp)
w_fine = np.linspace(0.5 * w0, 1.5 * w0, 200001)          # 在 w0 附近用细网格测量带宽
mag_fine = np.abs(R_bp / (R_bp + 1j * (w_fine * L_bp - 1 / (w_fine * C_bp))))
band = w_fine[mag_fine >= 1 / np.sqrt(2)]
print(f'带通: 手算 == scipy.freqs ? {np.allclose(H_bp, H_bp_scipy)}, 峰值频率 = {w_fine[np.argmax(mag_fine)]:.1f} (w0 = {w0:.1f}), '
      f'-3 dB 带宽 数值 = {band[-1] - band[0]:.1f}, R/L = {R_bp / L_bp:.1f} rad/s')

plt.figure(figsize=(4, 3))
plt.semilogx(w, 20 * np.log10(np.abs(H_lp)), label='RC low-pass')
plt.semilogx(w, 20 * np.log10(np.abs(H_hp)), label='RC high-pass')
plt.semilogx(w, 20 * np.log10(mag_bp), label='RLC band-pass')
plt.axhline(-3, color='gray', linestyle=':', label='-3 dB')
plt.ylim(-60, 5)
plt.xlabel('omega (rad/s)')
plt.ylabel('dB')
plot_nicely()
plt.show()

<a id="filtering"></a>

$$
\text{用 RC 低通滤波器去噪: 时域仿真} \\
\text{ODE: } \tau \frac{dv}{dt} + v = x(t) \text{ (x 是输入信号)} \\
\text{欧拉法离散化 (步长 h, 导数用差商代替): } \tau \frac{v_{n+1} - v_n}{h} + v_n = x_n \\
v_{n+1} = v_n + \frac{h}{\tau}(x_n - v_n) = (1 - \alpha) v_n + \alpha x_n, \quad \alpha = \frac{h}{\tau} \\
\text{这正是指数移动平均 (Exponential Moving Average, EMA) / 指数平滑: RC 低通滤波器的离散版本} \\
\text{(展开: } v_n = \alpha \sum_{k} (1 - \alpha)^k x_{n-1-k} \text{, 越早的数据权重按几何级数衰减)} \\ \quad \\
\text{例: 输入 = 5 Hz 有用信号 + 200 Hz 干扰 + 白噪声, 取 } f_c = 20 \text{ Hz} \\
\text{预测: 5 Hz 分量增益 } \frac{1}{\sqrt{1 + (5/20)^2}} \approx 0.970 \text{, 200 Hz 分量增益 } \frac{1}{\sqrt{1 + (200/20)^2}} \approx 0.0995 \\ \quad \\
\text{白噪声 (各频率功率相同) 经过滤波器后的方差 = 功率谱密度 } \times \int_0^\infty |H(f)|^2 df \\
\int_0^\infty \frac{df}{1 + (f / f_c)^2} \xrightarrow{u = f / f_c} f_c \int_0^\infty \frac{du}{1 + u^2} = f_c \left[\arctan u\right]_0^\infty = \frac{\pi}{2} f_c \text{ (等效噪声带宽)} \\
\text{采样率 } f_s \text{, 方差 } \sigma^2 \text{ 的离散白噪声, 单边功率谱密度 } = \frac{2\sigma^2}{f_s} \text{ (功率均匀分布在 } [0, \tfrac{f_s}{2}] \text{)} \\
\sigma_{out}^2 = \frac{2\sigma^2}{f_s} \cdot \frac{\pi}{2} f_c = \frac{\pi f_c \sigma^2}{f_s} \\ \quad \\
\text{用离散的 EMA 直接推导: 平稳时 } Var(v_{n+1}) = Var(v_n) = V \text{, 且 } v_n \text{ 和新的噪声 } x_n \text{ 独立} \\
V = (1 - \alpha)^2 V + \alpha^2 \sigma^2 \implies V = \frac{\alpha^2 \sigma^2}{1 - (1 - \alpha)^2} = \frac{\alpha \sigma^2}{2 - \alpha} \\
\alpha \text{ 很小时 } V \approx \frac{\alpha \sigma^2}{2} = \frac{h}{2\tau}\sigma^2 = \frac{2\pi f_c}{2 f_s}\sigma^2 = \frac{\pi f_c \sigma^2}{f_s} \text{, 两种推导一致}
$$

数学基础: [欧拉法](../math_basics/calculus.ipynb#euler-rk4) · [最小二乘](../math_basics/linear_algebra.ipynb#projection)

In [ ]:
rng = np.random.default_rng(0)
fs = 5000.0
h = 1 / fs
t = np.arange(0, 2.0, h)
f_sig, f_noise, f_cut = 5.0, 200.0, 20.0
tau_f = 1 / (2 * np.pi * f_cut)
sigma_noise = 0.2
x_clean = np.sin(2 * np.pi * f_sig * t) + 0.5 * np.sin(2 * np.pi * f_noise * t)
x = x_clean + sigma_noise * rng.normal(size=t.size)

# 1. 手写欧拉 / EMA
alpha = h / tau_f

def ema(x, alpha):
    v = np.zeros_like(x)
    for n in range(len(x) - 1):
        v[n + 1] = (1 - alpha) * v[n] + alpha * x[n]
    return v

v_euler = ema(x, alpha)

# 2. scipy.signal.lsim (连续系统 H(s) = 1 / (tau s + 1))
_, v_lsim, _ = signal.lsim(signal.lti([1], [tau_f, 1]), U=x, T=t)
print(f'alpha = h / tau = {alpha:.5f}, 欧拉 vs lsim 最大差 = {np.max(np.abs(v_euler - v_lsim)):.2e} (步长越小越接近)')

# 3. 测量输出中各频率分量的振幅 (稳态部分, 最小二乘拟合 cos / sin)
def amplitude(sig, t, f):
    M = np.column_stack([np.cos(2 * np.pi * f * t), np.sin(2 * np.pi * f * t), np.ones_like(t)])
    a, b, _ = np.linalg.lstsq(M, sig, rcond=None)[0]
    return np.hypot(a, b)

# 线性系统满足叠加原理: 输出 = 干净信号的输出 + 噪声的输出, 分开测量
_, v_clean, _ = signal.lsim(signal.lti([1], [tau_f, 1]), U=x_clean, T=t)
steady = t > 0.5
for f_k, amp_in in [(f_sig, 1.0), (f_noise, 0.5)]:
    gain_formula = 1 / np.sqrt(1 + (f_k / f_cut)**2)
    # lsim 把采样点之间的输入当成直线 (线性插值), 对频率 f 的正弦相当于多乘了 sinc^2(f / fs), 200 Hz 时约 0.995
    interp_factor = np.sinc(f_k / fs)**2
    print(f'{f_k:>5.0f} Hz: 输出振幅 = {amplitude(v_clean[steady], t[steady], f_k):.4f}, 预测 = {amp_in * gain_formula:.4f} (增益 {gain_formula:.4f}), '
          f'考虑线性插值后 = {amp_in * gain_formula * interp_factor:.4f}')

# 白噪声经过 EMA (离散 RC) 后的标准差: 输出相邻点高度相关 (相关长度约 tau/h 个点), 需要很长的序列才能估计准
noise_out = ema(sigma_noise * rng.normal(size=400000), alpha)[2000:]
print(f'白噪声标准差: 输入 = {sigma_noise:.4f}, EMA 输出 = {noise_out.std():.4f}, '
      f'离散公式 sqrt(alpha sigma^2 / (2 - alpha)) = {np.sqrt(alpha * sigma_noise**2 / (2 - alpha)):.4f}, '
      f'连续近似 sqrt(pi f_c sigma^2 / f_s) = {np.sqrt(np.pi * f_cut * sigma_noise**2 / fs):.4f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
sel = (t > 1.0) & (t < 1.4)
axs[0].plot(t[sel], x[sel], alpha=0.4, label='noisy input')
axs[0].plot(t[sel], v_lsim[sel], label='RC output (lsim)')
axs[0].plot(t[sel], v_euler[sel], '--', label='EMA (Euler)')
axs[0].set_xlabel('t (s)')
freq = np.fft.rfftfreq(steady.sum(), h)
axs[1].semilogy(freq, np.abs(np.fft.rfft(x[steady])), alpha=0.5, label='input spectrum')
axs[1].semilogy(freq, np.abs(np.fft.rfft(v_lsim[steady])), label='output spectrum')
axs[1].set_xlim(0, 300)
axs[1].set_xlabel('f (Hz)')
plot_nicely()
plt.show()